In [1]:
import pandas as pd
import re
import json
import math
import os
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

In [2]:
# =========================
# Settings
# =========================
FILE_PATH = "./data/c3_d_test.xlsx"

M1 = "qwen3:8b"
M2 = "glm4:9b"
M3 = "huihui_ai/internlm3-abliterated"

MODELS = [
    (M1, False),
    (M2, False),
    (M3, False)
]

THINK_MODEL = "qwen3:8b"
THINK_MODEL_THINK = True

BEST_MODEL_NAME = "qwen3:8b"

TEMPERATURE = 0.0

RESULTS_DIR = "./Results/d1/"

def get_out_filename(model_name):
    clean_name = model_name.replace("/", "_").replace(":", "_")
    return RESULTS_DIR + f"result_{clean_name}.xlsx"

OUT_MODEL_FILE = {
    M1: get_out_filename(M1),
    M2: get_out_filename(M2),
    M3: get_out_filename(M3)
}

OUT_ENSEMBLE_FILE = "result_ensemble_LLM7_d.xlsx"

OUT_SUMMARY_XLSX = "accuracy_summary_LLM7_d.xlsx"
OUT_SUMMARY_CSV = "accuracy_summary_LLM7_d.csv"

In [3]:
# =========================
# Ollama call
# =========================
def query_ollama(prompt, model, CONTEXT_LEN=1536, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": CONTEXT_LEN, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

In [4]:
# =========================
# Utility: safe cast / NaN handling
# =========================
def is_nan(x) -> bool:
    try:
        return x is not None and isinstance(x, float) and math.isnan(x)
    except Exception:
        return False

def to_int_1_4(x):
    if x is None or pd.isna(x):
        return None
    try:
        xi = int(x)
    except Exception:
        try:
            s = str(x).strip()
            if s in ["1", "2", "3", "4"]:
                return int(s)
            return None
        except Exception:
            return None
    return xi if xi in (1, 2, 3, 4) else None

def to_conf_0_1(x):
    if x is None or pd.isna(x):
        return 0.0
    try:
        xf = float(x)
    except Exception:
        return 0.0
    if math.isnan(xf):
        return 0.0
    return max(0.0, min(1.0, xf))

In [5]:
# =========================
# Parsing helpers
# =========================
def _strip_think(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def _normalize(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _best_option_match(text_cleaned: str, options: dict) -> int | None:
    t = _normalize(text_cleaned)
    if not t:
        return None
    for k, opt in options.items():
        o = _normalize(opt)
        if o and (o in t or t in o):
            return int(k)
    best_k, best_score = None, 0.0
    for k, opt in options.items():
        o = _normalize(opt)
        if not o:
            continue
        score = SequenceMatcher(None, t, o).ratio()
        if score > best_score:
            best_k, best_score = k, score
    if best_k is not None and best_score >= 0.55:
        return int(best_k)
    return None

def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    if not raw_text:
        return None, None
    text = _strip_think(raw_text)

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ans = to_int_1_4(obj.get("answer", None))
            conf = obj.get("confidence", None)
            conf = None if conf is None else float(conf)
            if conf is not None:
                conf = max(0.0, min(1.0, conf))
            if ans is not None:
                return ans, conf
        except Exception:
            pass

    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    m2 = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    conf = float(m2.group(1)) if m2 else None
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    if ans is not None:
        return ans, conf

    nums = re.findall(r"\b([1-4])\b", text)
    if nums:
        return int(nums[-1]), conf

    matched = _best_option_match(text, options)
    return matched, conf

In [6]:
# =========================
# Prompt builder
# =========================
def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
1) Select the best option among ({valid_numbers}). If an option is missing/empty, do not select it.
2) Output ONLY a JSON object with:
   - "answer": an integer in [{valid_numbers}]
   - "confidence": a number in [0, 1] representing your probability the answer is correct.

Example:
{{"answer": 2, "confidence": 0.78}}
""".strip()

# =========================
# Ensemble helpers
# =========================
def majority_vote(preds: dict) -> int | str | None:
    counts = {}
    for a in preds.values():
        a = to_int_1_4(a)
        if a is None:
            continue
        counts[a] = counts.get(a, 0) + 1
    if not counts:
        return None
    best_a = max(counts, key=counts.get)
    top = [k for k, v in counts.items() if v == counts[best_a]]
    if len(top) > 1:
        return "TIE"
    return best_a

def confidence_weighted_vote(preds: dict, confs: dict) -> int | str | None:
    scores = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    total_mass = 0.0
    for m, a in preds.items():
        a = to_int_1_4(a)
        if a is None:
            continue
        w = to_conf_0_1(confs.get(m, None))
        scores[a] += w
        total_mass += w
    if total_mass <= 0.0:
        return None
    best_a = max(scores, key=lambda k: scores[k])
    best_score = scores[best_a]
    tied = [k for k, v in scores.items() if abs(v - best_score) < 1e-12]
    if len(tied) > 1:
        return "TIE"
    return best_a

In [7]:
# =========================
# Step 1: Run one model over whole dataset
# =========================
def run_single_model(df: pd.DataFrame, model_name: str, think: bool) -> tuple[str, int, int, float]:
    out_file = OUT_MODEL_FILE.get(model_name, RESULTS_DIR + f"result_{model_name.replace(':', '_')}.xlsx")

    if os.path.exists(out_file):
        print(f"\n[Info] {model_name} 的輸出檔已存在 ({out_file})，直接讀取結果...")
        res_df = pd.read_excel(out_file)
        correct = int(res_df["Correct"].sum())
        total = len(res_df)
        acc = (correct / total * 100) if total else 0.0
        print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
        return out_file, correct, total, acc

    print(f"\n====== Running {model_name} (think={think}) ======")
    correct = 0
    results = []
    total = len(df)

    for _, row in tqdm(df.iterrows(), total=total, desc=f"Predicting[{model_name}]"):
        options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
        valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}

        prompt = build_prompt(row["Article"], row["Question"], valid_options)
        raw = query_ollama(prompt, model=model_name, CONTEXT_LEN=1536, think=think)

        pred, conf = parse_answer_and_confidence(raw, valid_options)
        pred = to_int_1_4(pred)

        actual = to_int_1_4(row["Answer"])
        is_correct = (actual is not None and pred is not None and pred == actual)
        if is_correct:
            correct += 1

        results.append({
            "ID": row["ID"], "Actual": actual, "Predicted": pred,
            "Confidence": conf, "Correct": is_correct, "Model_Response": raw
        })

    acc = (correct / total * 100) if total else 0.0
    print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")
    pd.DataFrame(results).to_excel(out_file, index=False)
    print(f"[Saved] {out_file}")
    return out_file, correct, total, acc

In [8]:
# =========================
# Step 2: Ensembles
# =========================
def query_thinking_with_retry(prompt, valid_options, max_retries=3):
    for attempt in range(max_retries):
        ds_raw = query_ollama(prompt, model=THINK_MODEL, CONTEXT_LEN=3192, think=THINK_MODEL_THINK)
        ds_pred, _ = parse_answer_and_confidence(ds_raw, valid_options)
        ds_pred = to_int_1_4(ds_pred)
        if ds_pred is not None:
            return ds_pred, False
    return None, True

def run_ensembles_separately(df_base: pd.DataFrame, outputs: dict):
    df_g = pd.read_excel(outputs[M1])
    df_q = pd.read_excel(outputs[M2])
    df_d = pd.read_excel(outputs[M3])

    total = len(df_base)

    actuals = df_g["Actual"].tolist()
    preds_g = df_g["Predicted"].tolist()
    preds_q = df_q["Predicted"].tolist()
    preds_d = df_d["Predicted"].tolist()
    confs_g = df_g["Confidence"].tolist()
    confs_q = df_q["Confidence"].tolist()
    confs_d = df_d["Confidence"].tolist()
    best_preds = df_g["Predicted"].tolist()

    print("C3_D Majority Voting + Thinking mode 結果如下:")

    maj_correct = 0
    maj_preds, maj_corrs = [], []
    maj_tie_count = 0
    maj_tie_all_fail = 0
    maj_tie_think_success = 0
    for i in tqdm(range(total), desc="Running [Majority Voting (MV)]"):
        row = df_base.iloc[i]
        preds = {M1: preds_g[i], M2: preds_q[i], M3: preds_d[i]}
        ans = majority_vote(preds)
        if ans == "TIE":
            maj_tie_count += 1
            options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
            valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}
            prompt = build_prompt(row["Article"], row["Question"], valid_options)
            ds_pred, all_failed = query_thinking_with_retry(prompt, valid_options)
            if all_failed:
                maj_tie_all_fail += 1
                ans = to_int_1_4(best_preds[i])
            else:
                ans = ds_pred
                maj_tie_think_success += 1
        is_corr = (ans is not None and ans == actuals[i])
        maj_preds.append(ans)
        maj_corrs.append(is_corr)
        if is_corr:
            maj_correct += 1

    maj_acc = (maj_correct / total * 100) if total else 0.0
    print(f"Majority Accuracy: {maj_correct}/{total} = {maj_acc:.2f}%")
    print(f"平手觸發 Thinking Model 次數: {maj_tie_count} 筆")
    print(f" ├─ Thinking Model 成功給出作答: {maj_tie_think_success} 筆")
    print(f" ├─ Thinking Model 重試後仍無答案 (採用最強單體模型): {maj_tie_all_fail} 筆")
    print(f" └─ Thinking Model 連跑3次都沒答案的次數: {maj_tie_all_fail} 筆")

    print("\nC3_D Confidence Weighted + Thinking mode 結果如下:")

    wv_correct = 0
    wv_preds, wv_corrs = [], []
    wv_tie_count = 0
    wv_tie_all_fail = 0
    wv_tie_think_success = 0
    for i in tqdm(range(total), desc="Running [Confidence-Weighted Voting (CWV)]"):
        row = df_base.iloc[i]
        preds = {M1: preds_g[i], M2: preds_q[i], M3: preds_d[i]}
        confs = {M1: confs_g[i], M2: confs_q[i], M3: confs_d[i]}
        ans = confidence_weighted_vote(preds, confs)
        if ans == "TIE":
            wv_tie_count += 1
            options_all = {1: row["Option1"], 2: row["Option2"], 3: row["Option3"], 4: row["Option4"]}
            valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}
            prompt = build_prompt(row["Article"], row["Question"], valid_options)
            ds_pred, all_failed = query_thinking_with_retry(prompt, valid_options)
            if all_failed:
                wv_tie_all_fail += 1
                ans = to_int_1_4(best_preds[i])
            else:
                ans = ds_pred
                wv_tie_think_success += 1
        is_corr = (ans is not None and ans == actuals[i])
        wv_preds.append(ans)
        wv_corrs.append(is_corr)
        if is_corr:
            wv_correct += 1

    wv_acc = (wv_correct / total * 100) if total else 0.0
    print(f"Weighted Accuracy: {wv_correct}/{total} = {wv_acc:.2f}%")
    print(f"平手觸發 Thinking Model 次數: {wv_tie_count} 筆")
    print(f" ├─ Thinking Model 成功給出作答: {wv_tie_think_success} 筆")
    print(f" └─ Thinking Model 連跑3次都沒答案的次數 (採用最強單體模型): {wv_tie_all_fail} 筆")

    df_ensemble = pd.DataFrame({
        "ID": df_base["ID"].tolist(),
        "Actual": actuals,
        "M1_Pred": preds_g,
        "M2_Pred": preds_q,
        "M3_Pred": preds_d,
        "MajorityVote": maj_preds,
        "MajorityVote_Correct": maj_corrs,
        "Confidence_WeightedVote": wv_preds,
        "Confidence_WeightedVote_Correct": wv_corrs,
    })
    df_ensemble.to_excel(OUT_ENSEMBLE_FILE, index=False)
    print(f"\n[Saved] {OUT_ENSEMBLE_FILE}")

    return (maj_correct, maj_acc), (wv_correct, wv_acc)

In [9]:
# =========================
# Entry
# =========================
print(f"Loading dataset: {FILE_PATH}")
df = pd.read_excel(FILE_PATH)

required_columns = ["ID", "Article", "Question", "Option1", "Option2", "Option3", "Option4", "Answer"]
if not all(c in df.columns for c in required_columns):
    raise ValueError(f"Excel 缺少必要欄位，需要: {required_columns}")

Loading dataset: ./data/c3_d_test.xlsx


In [10]:
outputs = {}
summary_rows = []
model_accuracies = {}

for model_name, think in MODELS:
    out_file, correct, total, acc = run_single_model(df, model_name, think)
    outputs[model_name] = out_file
    model_accuracies[model_name] = acc
    summary_rows.append({
        "Method": model_name,
        "Correct": correct,
        "Total": total,
        "Accuracy(%)": round(acc, 4)
    })


[Info] qwen3:8b 的輸出檔已存在 (./Results/d1/result_qwen3_8b.xlsx)，直接讀取結果...
qwen3:8b Accuracy: 1753/1890 = 92.75%

[Info] glm4:9b 的輸出檔已存在 (./Results/d1/result_glm4_9b.xlsx)，直接讀取結果...
glm4:9b Accuracy: 1752/1890 = 92.70%

[Info] huihui_ai/internlm3-abliterated 的輸出檔已存在 (./Results/d1/result_huihui_ai_internlm3-abliterated.xlsx)，直接讀取結果...
huihui_ai/internlm3-abliterated Accuracy: 1751/1890 = 92.65%


In [11]:
print("\n開始進行 Ensemble 計算...\n")

(maj_correct, maj_acc), (wv_correct, wv_acc) = run_ensembles_separately(
    df_base=df,
    outputs=outputs
)

summary_rows.extend([
    {"Method": "Ensemble-Majority (3 models)", "Correct": maj_correct, "Total": len(df), "Accuracy(%)": round(maj_acc, 4)},
    {"Method": "Ensemble-Confidence_Weighted (3 models)", "Correct": wv_correct, "Total": len(df), "Accuracy(%)": round(wv_acc, 4)},
])

summary_df = pd.DataFrame(summary_rows).sort_values(by="Accuracy(%)", ascending=False)
summary_df.to_excel(OUT_SUMMARY_XLSX, index=False)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"\n[Saved] {OUT_SUMMARY_XLSX}")
print(f"[Saved] {OUT_SUMMARY_CSV}")
print("\n===== Accuracy Summary =====")
print(summary_df.to_string(index=False))


開始進行 Ensemble 計算...

C3_D Majority Voting + Thinking mode 結果如下:


Running [Majority Voting (MV)]: 100%|████████████████████████████████████████████| 1890/1890 [1:20:02<00:00,  2.54s/it]


Majority Accuracy: 1781/1890 = 94.23%
平手觸發 Thinking Model 次數: 17 筆
 ├─ Thinking Model 成功給出作答: 16 筆
 ├─ Thinking Model 重試後仍無答案 (採用最強單體模型): 1 筆
 └─ Thinking Model 連跑3次都沒答案的次數: 1 筆

C3_D Confidence Weighted + Thinking mode 結果如下:


Running [Confidence-Weighted Voting (CWV)]: 100%|██████████████████████████████████| 1890/1890 [03:16<00:00,  9.61it/s]


Weighted Accuracy: 1777/1890 = 94.02%
平手觸發 Thinking Model 次數: 8 筆
 ├─ Thinking Model 成功給出作答: 8 筆
 └─ Thinking Model 連跑3次都沒答案的次數 (採用最強單體模型): 0 筆

[Saved] result_ensemble_LLM7_d.xlsx

[Saved] accuracy_summary_LLM7_d.xlsx
[Saved] accuracy_summary_LLM7_d.csv

===== Accuracy Summary =====
                                 Method  Correct  Total  Accuracy(%)
           Ensemble-Majority (3 models)     1781   1890      94.2328
Ensemble-Confidence_Weighted (3 models)     1777   1890      94.0212
                               qwen3:8b     1753   1890      92.7513
                                glm4:9b     1752   1890      92.6984
        huihui_ai/internlm3-abliterated     1751   1890      92.6455
